<a href="https://colab.research.google.com/github/bahadursk/Py1/blob/master/hbond_counts_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting H-Bond Donor & Acceptor Counts from SMILES

**Goal:** given only a SMILES string, predict the number of hydrogen bond donors
and hydrogen bond acceptors — two of the four properties in Lipinski's Rule of
Five (along with molecular weight and LogP), important for judging a molecule's
likely oral bioavailability.

**Important honesty note:** unlike XLogP (a proprietary empirical estimate),
H-Bond Donor/Acceptor counts are **directly countable from structure** — RDKit
has built-in functions (`NumHDonors`, `NumHAcceptors`) that compute them exactly,
instantly, no training required. So this notebook builds a machine learning
model for consistency with your other workflows and as a learning exercise, but
also shows you the direct RDKit calculation alongside it — in real use, prefer
the direct calculation; the ML model is mainly useful for sanity-checking
PubChem's exact counting convention, or if you ever need to predict from a
structure representation other than SMILES.

**Data:** `PubChem_compound_anticancer.csv` — all 216 compounds have both counts
filled in, so no rows need to be dropped for missing labels.

## 1. Install & import dependencies

In [1]:
!pip install rdkit -q

import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

print("Imports OK")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 68.3 MB/s eta 0:00:00
Imports OK


## 2. Load your anticancer CSV\n\nLooks for the file directly in Colab's local storage (case-insensitive).

In [2]:
candidates = [f for f in glob.glob("*") if f.lower() == "pubchem_compound_anticancer.csv"]

if not candidates:
    raise FileNotFoundError(
        "Could not find 'PubChem_compound_anticancer.csv' in the current directory. "
        "Check the folder icon on the left sidebar, or upload it with:\n"
        "    from google.colab import files\n"
        "    uploaded = files.upload()"
    )

anticancer_path = Path(candidates[0])
print("Using file:", anticancer_path)

df_raw = pd.read_csv(anticancer_path)
print("Rows:", len(df_raw))


Using file: PubChem_compound_anticancer.csv
Rows: 216


## 3. Prepare targets\n\nBoth `H-Bond_Donor_Count` and `H-Bond_Acceptor_Count` are fully populated (0 missing), so we keep all rows this time.

In [3]:
df = df_raw[["SMILES", "Name", "H-Bond_Donor_Count", "H-Bond_Acceptor_Count"]].copy()
df = df.dropna(subset=["SMILES", "H-Bond_Donor_Count", "H-Bond_Acceptor_Count"])
df["H-Bond_Donor_Count"] = pd.to_numeric(df["H-Bond_Donor_Count"], errors="coerce")
df["H-Bond_Acceptor_Count"] = pd.to_numeric(df["H-Bond_Acceptor_Count"], errors="coerce")
df = df.dropna(subset=["H-Bond_Donor_Count", "H-Bond_Acceptor_Count"])
print(f"Labeled compounds available: {len(df)} / {len(df_raw)}")

df[["H-Bond_Donor_Count", "H-Bond_Acceptor_Count"]].describe()


Labeled compounds available: 216 / 216


,H-Bond_Donor_Count,H-Bond_Acceptor_Count
count,216.000000,216.000000
mean,1.467593,6.291667
std,1.200336,2.868129
min,0.000000,1.000000
25%,1.000000,4.000000
50%,1.000000,6.000000
75%,2.000000,8.000000
max,7.000000,16.000000


## 4. Compute RDKit descriptors from SMILES\n\n**Important:** we deliberately do NOT include RDKit's own `NumHDonors` / `NumHAcceptors` as input features here — that would make the model just copy an answer it already has access to, rather than genuinely learning a relationship from other structural properties. The features below are everything *except* those two.

In [4]:
FEATURE_NAMES = [
    "MolWt", "MolLogP_Crippen", "TPSA", "NumRotatableBonds",
    "RingCount", "NumAromaticRings", "FractionCSP3",
    "HeavyAtomCount", "NumHeteroatoms", "NumAliphaticRings",
]

def get_features(smiles: str):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return [
            Descriptors.MolWt(mol),
            Crippen.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumRotatableBonds(mol),
            Descriptors.RingCount(mol),
            Descriptors.NumAromaticRings(mol),
            Descriptors.FractionCSP3(mol),
            Descriptors.HeavyAtomCount(mol),
            Descriptors.NumHeteroatoms(mol),
            Descriptors.NumAliphaticRings(mol),
        ]
    except Exception:
        return None

# Separately: the DIRECT, exact calculation (for comparison later)
def get_exact_hbond_counts(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    return Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol)

df["features"] = df["SMILES"].apply(get_features)
n_before = len(df)
df = df.dropna(subset=["features"])
print(f"Valid molecules after RDKit parsing: {len(df)} / {n_before}")


Valid molecules after RDKit parsing: 216 / 216


## 5. Train/test split + scaling\n\nOne split, shared for both targets, so results are directly comparable.

In [5]:
X_all = np.array(list(df["features"]), dtype=np.float64)
y_donor = df["H-Bond_Donor_Count"].values.astype(np.float64)
y_acceptor = df["H-Bond_Acceptor_Count"].values.astype(np.float64)

# stack both targets into one 2-column array for a single train/test split
y_all = np.column_stack([y_donor, y_acceptor])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)


Train: (172, 10)  Test: (44, 10)


## 6. Random Forest (multi-output)\n\nscikit-learn's `RandomForestRegressor` can predict multiple targets at once when you give it a 2-column `y` — it trains one forest that outputs both the donor count and acceptor count together.

In [6]:
rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)

cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
# cross_val_score only supports one target at a time, so we check each column separately
cv_donor = cross_val_score(rf, X_train_s, y_train[:, 0], cv=cv, scoring="r2")
cv_acceptor = cross_val_score(rf, X_train_s, y_train[:, 1], cv=cv, scoring="r2")
print(f"5-fold CV R^2 - Donor count:    {cv_donor.mean():.3f} +/- {cv_donor.std():.3f}")
print(f"5-fold CV R^2 - Acceptor count: {cv_acceptor.mean():.3f} +/- {cv_acceptor.std():.3f}")

rf.fit(X_train_s, y_train)
test_preds = rf.predict(X_test_s)  # shape: (n_test, 2)

for i, label in enumerate(["Donor", "Acceptor"]):
    rmse = mean_squared_error(y_test[:, i], test_preds[:, i]) ** 0.5
    mae = mean_absolute_error(y_test[:, i], test_preds[:, i])
    r2 = r2_score(y_test[:, i], test_preds[:, i])
    print(f"\n{label} count - held-out test set:")
    print(f"  RMSE: {rmse:.3f} | MAE: {mae:.3f} | R^2: {r2:.3f}")


5-fold CV R^2 - Donor count:    0.349 +/- 0.123
5-fold CV R^2 - Acceptor count: 0.762 +/- 0.040

Donor count - held-out test set:
  RMSE: 1.206 | MAE: 0.889 | R^2: 0.132

Acceptor count - held-out test set:
  RMSE: 1.105 | MAE: 0.873 | R^2: 0.823


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, (ax, label) in enumerate(zip(axes, ["H-Bond Donor Count", "H-Bond Acceptor Count"])):
    ax.scatter(y_test[:, i], test_preds[:, i], alpha=0.7)
    lims = [min(y_test[:, i].min(), test_preds[:, i].min()) - 0.5,
            max(y_test[:, i].max(), test_preds[:, i].max()) + 0.5]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel(f"Actual {label}")
    ax.set_ylabel(f"Predicted {label}")
    ax.set_title(label)
plt.tight_layout()
plt.show()

# Feature importance for each target
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for i, (ax, label) in enumerate(zip(axes, ["Donor", "Acceptor"])):
    importances = pd.Series(rf.estimators_[0].feature_importances_, index=FEATURE_NAMES)
    # use the whole-forest importance (averaged across trees, same for both outputs
    # since it's one shared forest) - shown once is sufficient, but plotted per-panel for clarity
    importances = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values()
    importances.plot(kind="barh", ax=ax)
    ax.set_title(f"Feature importance (shared forest)")
plt.tight_layout()
plt.show()


## 7. Predict for unknown molecules — ML prediction vs. direct RDKit calculation\n\nBoth are shown side by side. **In practice, trust the direct RDKit column** (`exact_...`) — it's the true count, not an estimate. The ML columns are included to show how close a model *can* get without being told the exact rule.

In [9]:
unknown_smiles = [
    "OC1=CC=CC=C1",          # example: aspirin - replace with your own SMILES
    "CC1=CC(C2=CC=CC=C2)=CC=C1",       # example: caffeine
]

def predict_hbond_counts(smiles_list):
    results = []
    for smi in smiles_list:
        feats = get_features(smi)
        exact_donor, exact_acceptor = get_exact_hbond_counts(smi)
        if feats is None:
            results.append({
                "SMILES": smi, "predicted_donor_count": None, "predicted_acceptor_count": None,
                "exact_donor_count": None, "exact_acceptor_count": None,
                "note": "Invalid SMILES - could not parse"
            })
            continue
        feats_scaled = scaler.transform([feats])
        pred = rf.predict(feats_scaled)[0]
        results.append({
            "SMILES": smi,
            "predicted_donor_count": round(float(pred[0])),
            "predicted_acceptor_count": round(float(pred[1])),
            "exact_donor_count": exact_donor,
            "exact_acceptor_count": exact_acceptor,
            "note": "predicted_* = ML estimate, exact_* = true RDKit count",
        })
    return pd.DataFrame(results)

predictions_df = predict_hbond_counts(unknown_smiles)
predictions_df


,SMILES,predicted_donor_count,predicted_acceptor_count,exact_donor_count,exact_acceptor_count,note
0,OC1=CC=CC=C1,1,2,1,1,"predicted_* = ML estimate, exact_* = true RDKi..."
1,CC1=CC(C2=CC=CC=C2)=CC=C1,1,2,0,0,"predicted_* = ML estimate, exact_* = true RDKi..."


### Applicability domain check\n\nFlags molecules whose descriptors fall outside your training set's range — relevant only for the ML predictions, since the exact RDKit counts are always correct.

In [10]:
train_min = X_train.min(axis=0)
train_max = X_train.max(axis=0)

def in_domain(smiles):
    feats = get_features(smiles)
    if feats is None:
        return None
    feats = np.array(feats)
    return bool(((feats >= train_min) & (feats <= train_max)).all())

predictions_df["in_applicability_domain"] = predictions_df["SMILES"].apply(in_domain)
predictions_df


,SMILES,predicted_donor_count,predicted_acceptor_count,exact_donor_count,exact_acceptor_count,note,in_applicability_domain
0,OC1=CC=CC=C1,1,2,1,1,"predicted_* = ML estimate, exact_* = true RDKi...",False
1,CC1=CC(C2=CC=CC=C2)=CC=C1,1,2,0,0,"predicted_* = ML estimate, exact_* = true RDKi...",False


## 8. Save predictions

In [11]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
RUN_DIR = OUTPUT_DIR / f"run_{datetime.now().strftime('%Y%m%d-%H%M%S-%f')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

out_path = RUN_DIR / "hbond_count_predictions.csv"
predictions_df.to_csv(out_path, index=False)
print(f"Predictions saved to: {out_path}")

from google.colab import files
files.download(str(out_path))


Predictions saved to: outputs/run_20260712-070037-803051/hbond_count_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>